In [2]:
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from dgl.data import TUDataset
import time
from torch.utils.data import DataLoader, TensorDataset, random_split

# Parameters (Matching your original notebook)
N_MAX = 28 
INPUT_DIM = N_MAX # PointNet processes rows (N_MAX) of features (N_MAX)
OUTPUT_BASIS = 15          
OUTPUT_DIM = OUTPUT_BASIS * N_MAX * N_MAX 

dataset = TUDataset('MUTAG')
print(f"Dataset loaded: {len(dataset)} graphs.")

Dataset loaded: 188 graphs.


In [3]:
def ops_2_to_2(inputs, dim): 
    diag_part = th.diagonal(inputs, dim1=2, dim2=3) 
    sum_diag_part = th.sum(diag_part, dim=2, keepdim=True) 
    sum_of_rows = th.sum(inputs, dim=3) 
    sum_of_cols = th.sum(inputs, dim=2) 
    sum_all = th.sum(sum_of_rows, dim=2) 

    # Basis generation (Simplified for training targets)
    ops = [
        th.diag_embed(diag_part),
        th.diag_embed(sum_diag_part.repeat(1, 1, dim)),
        th.diag_embed(sum_of_rows),
        th.diag_embed(sum_of_cols),
        th.diag_embed(sum_all.unsqueeze(2).repeat(1, 1, dim)),
        sum_of_cols.unsqueeze(3).repeat(1, 1, 1, dim),
        sum_of_rows.unsqueeze(3).repeat(1, 1, 1, dim),
        sum_of_cols.unsqueeze(2).repeat(1, 1, dim, 1),
        sum_of_rows.unsqueeze(2).repeat(1, 1, dim, 1),
        inputs,
        th.transpose(inputs, -2, -1),
        diag_part.unsqueeze(3).repeat(1, 1, 1, dim),
        diag_part.unsqueeze(2).repeat(1, 1, dim, 1),
        sum_diag_part.unsqueeze(3).repeat(1, 1, dim, dim),
        sum_all.unsqueeze(2).unsqueeze(3).repeat(1, 1, dim, dim)
    ]
    return ops

In [4]:
class PointNetParrot(nn.Module):
    def __init__(self, node_dim=N_MAX, output_dim=OUTPUT_DIM):
        super(PointNetParrot, self).__init__()
        # MLP applied to each node/row independently
        self.phi = nn.Sequential(
            nn.Linear(node_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256)
        )
        # MLP applied to the Global Feature Vector
        self.rho = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim),
            nn.Sigmoid() # Helps with normalized targets
        )

    def forward(self, x):
        # x: [Batch, N_MAX, N_MAX]
        batch_size = x.size(0)
        
        # Step 1: Extract local features for every node
        x = self.phi(x) # Shape: [Batch, N_MAX, 256]
        
        # Step 2: Symmetric Aggregation (The GI Solver)
        # No matter the order of N_MAX, the Max is the same.
        global_feat, _ = th.max(x, dim=1) # Shape: [Batch, 256]
        
        # Step 3: Global mapping to 15 Basis Tensors
        out = self.rho(global_feat) 
        return out

In [5]:
def collect_pointnet_data(dataset, num_permutations=50):
    inputs_list = []
    outputs_list = []
    
    for graph, _ in dataset:
        adj = graph.adjacency_matrix().to_dense()
        num_nodes = adj.shape[0]
        
        # Get Canonical Math Truth
        padded_base = th.zeros((1, 1, N_MAX, N_MAX))
        padded_base[0, 0, :num_nodes, :num_nodes] = adj
        with th.no_grad():
            basis_list = ops_2_to_2(padded_base, N_MAX)
            target = th.stack(basis_list, dim=0).squeeze()
            
        for _ in range(num_permutations):
            perm = th.randperm(num_nodes)
            shuffled_adj = adj[perm][:, perm]
            padded_input = th.zeros((N_MAX, N_MAX))
            padded_input[:num_nodes, :num_nodes] = shuffled_adj
            
            inputs_list.append(padded_input) 
            outputs_list.append(target) # All shuffles map to same target
            
    return th.stack(inputs_list), th.stack(outputs_list)

raw_inputs, raw_outputs = collect_pointnet_data(dataset)
# Normalize outputs (matching your original logic)
y_flat = raw_outputs.view(raw_outputs.size(0), -1)
y_norm = (y_flat - y_flat.min()) / (y_flat.max() - y_flat.min() + 1e-7)

train_loader = DataLoader(TensorDataset(raw_inputs, y_norm), batch_size=32, shuffle=True)

In [6]:
model = PointNetParrot()
optimizer = th.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training Loop
model.train()
for epoch in range(50):
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0: print(f"Epoch {epoch} Loss: {loss.item():.6f}")

# GI Symmetry Test
model.eval()
test_graph, _ = dataset[0]
adj = test_graph.adjacency_matrix().to_dense()
num_nodes = adj.shape[0]

# Original vs Shuffle
in_a = th.zeros((1, N_MAX, N_MAX))
in_a[0, :num_nodes, :num_nodes] = adj

perm = th.randperm(num_nodes)
in_b = th.zeros((1, N_MAX, N_MAX))
in_b[0, :num_nodes, :num_nodes] = adj[perm][:, perm]

with th.no_grad():
    out_a = model(in_a)
    out_b = model(in_b)

inv_error = th.abs(out_a - out_b).mean().item()
print(f"\nPointNet Invariance Error: {inv_error:.10f}")
print("SUCCESS" if inv_error < 1e-5 else "FAILURE")

Epoch 0 Loss: 0.000537
Epoch 10 Loss: 0.000043
Epoch 20 Loss: 0.000046
Epoch 30 Loss: 0.000023
Epoch 40 Loss: 0.000022

PointNet Invariance Error: 0.0005244440
FAILURE


In [8]:
# 1. Setup the Parrot function to match expectations
def parrot_pointnet_ops(inputs, winner_model, y_min, y_max):
    # inputs: [1, 7, 28, 28] -> PointNet only needs the adjacency [1, 28, 28]
    adj_slice = inputs[:, 0, :, :] 
    
    with th.no_grad():
        y_pred_norm = winner_model(adj_slice)
    
    # Denormalize
    y_denorm = y_pred_norm * (y_max - y_min + 1e-7) + y_min
    y_final = y_denorm.view(15, N_MAX, N_MAX)
    
    # Return as a list of 1x1x28x28 tensors
    return [mat.unsqueeze(0).unsqueeze(0) for mat in y_final]

# 2. Run the Comparison Loop
num_tests = 10
total_error = 0
orig_times, parr_times = [], []

y_min = y_flat.min()
y_max = y_flat.max()

print(f"{'Graph ID':<10} | {'Orig Time (s)':<15} | {'Parr Time (s)':<15} | {'Error'}")
print("-" * 65)

for i in range(num_tests):
    test_graph, _ = dataset[i]
    adj = test_graph.adjacency_matrix().to_dense()
    num_nodes = adj.shape[0]
    
    # Ring-GNN uses 7 channels, but the basis math only uses the 1st (adj)
    # We create a 4D tensor [Batch, Channel, N, N] for the original math function
    test_in_4d = th.zeros((1, 1, N_MAX, N_MAX))
    test_in_4d[0, 0, :num_nodes, :num_nodes] = adj
    
    # Time Original Math (The 15 basis operators)
    t0 = time.time()
    out_orig = ops_2_to_2(test_in_4d, N_MAX)
    orig_times.append(time.time() - t0)
    
    # Time PointNet Parrot
    # We pass the same 4D input; the parrot function will slice it internally
    t0 = time.time()
    out_parr = parrot_pointnet_ops(test_in_4d, model, y_min, y_max)
    parr_times.append(time.time() - t0)
    
    # FIX: Stack both results into [15, 28, 28] tensors for subtraction
    # We use .squeeze() carefully to ensure shapes are [15, 28, 28]
    orig_tensor = th.cat(out_orig, dim=0).squeeze() # Combine list into one tensor
    parr_tensor = th.cat(out_parr, dim=0).squeeze()
    
    err = th.abs(orig_tensor - parr_tensor).mean().item()
    total_error += err
    
    print(f"{i:<10} | {orig_times[-1]:<15.6f} | {parr_times[-1]:<15.6f} | {err:.8f}")

print("-" * 65)
print(f"Final Speedup: {np.mean(orig_times)/np.mean(parr_times):.2f}x")
print(f"Final Avg Signature Error: {total_error/num_tests:.8f}")

Graph ID   | Orig Time (s)   | Parr Time (s)   | Error
-----------------------------------------------------------------
0          | 0.000141        | 0.001071        | 0.08511104
1          | 0.000110        | 0.000720        | 0.08445588
2          | 0.000109        | 0.000468        | 0.07382294
3          | 0.000107        | 0.000451        | 0.18233518
4          | 0.000111        | 0.000425        | 0.06478617
5          | 0.000114        | 0.000417        | 0.36287776
6          | 0.000105        | 0.000434        | 0.13471498
7          | 0.000104        | 0.000421        | 0.21511401
8          | 0.000103        | 0.000429        | 0.09546006
9          | 0.000100        | 0.000415        | 0.17469139
-----------------------------------------------------------------
Final Speedup: 0.21x
Final Avg Signature Error: 0.14733694
